In [1]:
import numpy as np
from collections import Counter


In [2]:
#encoding the dataset
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
labels=[]
for row in data:
    i=row[3]
    labels.append(i)
  
def encoding(col):
    labels1=[]
    for i in col:
        if i =='Wine':
            labels1.append(0)
        if i =='Beer':
            labels1.append(1)
        if i =='Whiskey':
            labels1.append(2)
    return labels1
    
X=np.array([row[:3] for row in data])
y=np.array(encoding(labels))
X,y

(array([[12. ,  1.5,  1. ],
        [ 5. ,  2. ,  0. ],
        [40. ,  0. ,  1. ],
        [13.5,  1.2,  1. ],
        [ 4.5,  1.8,  0. ],
        [38. ,  0.1,  1. ],
        [11.5,  1.7,  1. ],
        [ 5.5,  2.3,  0. ]]),
 array([0, 1, 2, 0, 1, 2, 0, 1]))

In [3]:
#implementing Gini impurity
def gini(y):
    n=len(y)
    counts=Counter(y)#Counter function returns a map which has labels and their values(frequencies)
    gini=1
    for i in counts.values():
        p=i/n
        gini-=p**2
    return gini

In [4]:
#bonus 1
def entropy(y):
    value,counts=Counter(y)
    probs=counts/len(y)

    return -np.sum(probs*np.log2(probs))

In [5]:
#best split 
def best_split(X,y):
    best_gain=-1
    best_feature=None
    best_threshold=None
    n=len(y)
    #logic:- we will just brute force every element(threshold) in every threshold
    for feature in range(X.shape[1]):#.shape[1] returns all the columns which are features here.
        thresholds=np.unique(X[:,feature])#just removing duplicates
        n=len(X)
        for threshold in thresholds:
            l=X[:,feature]<threshold #condition for left split
            left_y=y[l]
            r=X[:,feature]>=threshold #condition for right split
            right_y=y[r]
            n_left=len(left_y)
            n_right=len(right_y)
            weighted_gini=(n_left*(gini(left_y))+n_right*(gini(right_y)))/(n)
            gain=gini(y)-weighted_gini #gain is the measure of quality of split.
            if gain>best_gain:
                best_gain=gain
                best_feature=feature
                best_threshold=threshold
    return best_feature, best_threshold, best_gain

In [6]:
#creating a class to store attributes 
class Node:
    def __init__(self):
        self.feature_index=None
        self.threshold=None
        self.left=None
        self.right=None
        self.value=None #value is None only if the node is not a leaf
    def is_leaf(self):
        if self.value == None:
            return False
        return True

In [7]:
#Implementing Recursive Tree Building
def build_tree(X,y,depth=0,max_depth=5):##bonus 2 :- used max depth
    
    X=np.array(X)
    y=np.array(y)
    #condition 1 :- if all x in X has same label ( belongs to same class)
    
    if len(set(y)) == 1:
        leaf = Node()
        leaf.value = y[0]   # only one class, return it here we are giving some value to leaf this confirm that it is a leaf node
        return leaf
        
    #condition 2 :- if depth reached maax_depth

    if depth >= max_depth:
        leaf = Node()
        counts=Counter(y)
        max_count=-1;
        max_label=None
        for l,c in counts.items():#finding the majority class
            if c>max_count:
                max_count=c
                max_label=l
        leaf.value = max_label # majority class
        return leaf
        
    #condition 3 :-Number of samples is below a min threshold
    
    #here as min_threshold is 2 there can be only 1 element in X so we are dirctly returning label of that one X
    min_threshold=2
    if X.shape[0]<min_threshold:
        leaf=Node()
        leaf.value=y
        return leaf
     
    feature, threshold, gain = best_split(X, y)
    left_split_criteria  = X[:, feature] <  threshold
    right_split_criteria = X[:, feature] >= threshold
    
     
    left_child  = build_tree(X[left_split_criteria],  y[left_split_criteria],  depth+1, max_depth)
    right_child = build_tree(X[right_split_criteria], y[right_split_criteria], depth+1, max_depth)
   
    node = Node() #this node is the connection of the two child nodes          
    node.feature   = feature     # store WHICH column to check 
    node.threshold = threshold   # store the threshold value 
    node.left      = left_child  # attach left child
    node.right     = right_child # attach right child
    
    return node             

                    
    

In [8]:
#implementing prediction
#predict for one single x (recursively)
def predict_one(node,x):
    if node.is_leaf():
        return node.value
    if x[node.feature]<node.threshold:
        return predict_one(node.left,x)
    else:
        return predict_one(node.right,x)
#generalizing to complete X
def predict(node,X):
    results=[]
    for x in X:
        results.append(predict_one(node,x))
    return results
                

In [9]:
def decoding(col):
    labels=[]
    for i in col:
        if i ==0:
            labels.append('Wine')
        if i ==1:
            labels.append('Beer')
        if i ==2:
            labels.append('Whiskey')
    return labels
    

In [10]:
tree = build_tree(X, y)
predictions = predict(tree, X)
print(predictions)

[np.int64(0), np.int64(1), np.int64(2), np.int64(0), np.int64(1), np.int64(2), np.int64(0), np.int64(1)]


In [11]:
#evaluation 
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
    
X_test=np.array([row[:3] for row in test_data])

predictions_test = predict(tree, X_test)
print(decoding(predictions_test))

['Beer', 'Whiskey', 'Wine']
